# Fatigue modeling

Ordinal and classification models with participant-level held-out test, GroupKFold CV, and Optuna tuning. Core logic lives in `src/modeling/`.

Tuning and CV use **train/val participants only**; held-out test participants never appear in Optuna or CV folds.


In [41]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [42]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Ensure local src edits are picked up when re-running this cell.
for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

import pandas as pd
from modeling.baselines import (
    run_all_baseline_benchmarks,
    summarize_baseline_metrics,
)
from modeling.config import (
    DATA_PATH,
    HIGH_FATIGUE_THRESHOLD,
    N_CV_FOLDS,
    OPTUNA_TRIALS,
    STABILITY_SEEDS,
)
from modeling.data import load_fatigue_data, prepare_splits, split_summary_table
from modeling.registry import (
    CLASSIFICATION_MODELS,
    HISTORY_CLASSIFICATION_MODELS,
    HISTORY_ORDINAL_MODELS,
    ORDINAL_MODELS,
    RESIDUAL_CLASSIFICATION_MODELS,
    RESIDUAL_ORDINAL_MODELS,
)
from modeling.runner import tune_and_benchmark_model
from modeling.validation import run_stability_study, summarize_stability


## 1. Load data and split

Participants are held out with a **stratified split** on per-participant high-fatigue rate (`prepare_splits(..., stratify=True)`) so train/val and test have similar class balance.

“We randomly assign whole participants to train/val or test, but we do it in a way that both groups contain a similar proportion of people who often report high fatigue — not just a random 8 people who might all happen to be high-fatigue reporters.”


In [43]:
df = load_fatigue_data('../../' + DATA_PATH)
bundle = prepare_splits(df)
y_high_fatigue = (df['fatigue_num'] >= HIGH_FATIGUE_THRESHOLD).astype(int)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle, y_high_fatigue))
print('Test participant ids:', sorted(bundle.test_ids))


Rows: 3,331  Participants: 42


,split,participants,rows,mean_fatigue,high_fatigue_rate
0,train_val,34,2646,2.419123,0.267196
1,test,8,685,2.816058,0.319708


Test participant ids: [np.int64(10), np.int64(18), np.int64(30), np.int64(37), np.int64(38), np.int64(42), np.int64(46), np.int64(50)]


Re-run the **init accumulators** cell below before a fresh partial run to clear prior tuned-model results.


In [44]:
# Re-run this cell to clear accumulated model results before a fresh partial run.
ordinal_results = []
classification_results = []
history_ordinal_results = []

ordinal_best_params = {}
classification_best_params = {}
history_best_params = {}

## 2. Baseline benchmarks

Simple predictors evaluated with the same GroupKFold CV and held-out test protocol as the tuned models. Includes persistence baselines **`lag1_fatigue`** and **`expanding_mean`**.


In [45]:
ordinal_baseline_results = run_all_baseline_benchmarks(bundle, task='ordinal', n_splits=N_CV_FOLDS)
classification_baseline_results = run_all_baseline_benchmarks(bundle, task='classification', n_splits=N_CV_FOLDS)

ordinal_baseline_summary = summarize_baseline_metrics(ordinal_baseline_results, task='ordinal')
clf_baseline_summary = summarize_baseline_metrics(classification_baseline_results, task='classification')

print('Ordinal baselines (test metrics)')
display(ordinal_baseline_summary[[c for c in ordinal_baseline_summary.columns if c.startswith('test_')]])
print('Classification baselines (test metrics)')
display(clf_baseline_summary[[c for c in clf_baseline_summary.columns if c.startswith('test_')]])

Ordinal baselines (test metrics)


,test_mae,test_rmse,test_r2,test_qwk
model,,,,
global_mean,1.321168,1.585979,-0.360095,0.000000
global_mode,0.986861,1.372302,-0.018295,0.000000
lag1_fatigue,0.934307,1.362694,-0.004086,0.495885
expanding_mean,0.839416,1.167066,0.263512,0.458234


Classification baselines (test metrics)


,test_accuracy,test_f1,test_precision,test_recall
model,,,,
majority_class,0.680292,0.000000,0.000000,0.000000
lag1_high_fatigue,0.705109,0.534562,0.539535,0.529680
expanding_high_fatigue_rate,0.747445,0.548303,0.640244,0.479452


MAE: Mean Absolute Error;

RMSE: Root Mean Squared Error, measures the variation in residual/error

R2: how much variability is explained by the model

QWK: Quadratic Weighted Kappa. QWK measures the agreement between two raters—such as an AI and a human—on an ordered scale. It is designed to adjust for chance agreements and heavily penalize larger scoring discrepancies over minor ones.

## 3. Train/Tune models

Run only the model cells you need. Each cell tunes (Optuna) and benchmarks one model. Skip slow models like `lstm` unless required.


### Ordinal

#### `ordered_logistic`


In [26]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic  test_mae=1.3051


#### `ordinal_rf`


In [27]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf  test_mae=1.0657


#### `catboost_ordinal`


In [28]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal  test_mae=1.0496


#### `mixed_effects`


In [29]:
_name = 'mixed_effects'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] mixed_effects  test_mae=1.2146


#### `lstm` (slow)

In [12]:
# _name = 'lstm'
# _result, _params = tune_and_benchmark_model(
#     _name,
#     bundle,
#     ORDINAL_MODELS,
#     task='ordinal',
#     n_trials=OPTUNA_TRIALS,
#     n_splits=N_CV_FOLDS,
# )
# ordinal_results.append(_result)
# ordinal_best_params[_name] = _params
# print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')

### Classification

Specifically, we use F1 as our metric because of the class imbalance we have. 0 (worst) - 1 (best)

#### `lightgbm`


In [30]:
_name = 'lightgbm'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    CLASSIFICATION_MODELS,
    task='classification',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
classification_results.append(_result)
classification_best_params[_name] = _params
print(f'[ok] {_name}  test_f1={_result["test_metrics"]["f1"]:.4f}')


[ok] lightgbm  test_f1=0.4299


#### `random_forest`


In [31]:
_name = 'random_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    CLASSIFICATION_MODELS,
    task='classification',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
classification_results.append(_result)
classification_best_params[_name] = _params
print(f'[ok] {_name}  test_f1={_result["test_metrics"]["f1"]:.4f}')


[ok] random_forest  test_f1=0.4391


### History

History models share the same leakage-safe history features. **Ordinal** models predict `fatigue_num` (MAE); **classification** models predict `high_fatigue` (F1). Compare classification models vs `lag1_high_fatigue` and `expanding_high_fatigue_rate` baselines from section 1b.


Added **History features** (7 cols):
- fatigue lag1: Yesterday’s fatigue score
- fatigue EWMA: Exponentially weighted average of past fatigue; recent days count more
- fatigue expanding mean: Average fatigue on all earlier days for this person
- fatigue delta lag1: Change in fatigue, the worsening/improving trend
- activity_logsum_roll3_mean: Rolling mean of prior days' sum of log1p(lightly) + log1p(moderately) + log1p(very)
- calories_sum_roll3_mean: Recent typical daily calories burned
- very_roll3_mean: Recent typical “very active” minutes

#### `catboost_history`


In [46]:
_name = 'catboost_history'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    HISTORY_ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')
print(f'  params={_params}')
if _result.get('history_params'):
    print(f'  history_params={_result["history_params"]}')


[ok] catboost_history  test_mae=0.8321
  params={'iterations': 308, 'depth': 4, 'learning_rate': 0.029607929025176408, 'l2_leaf_reg': 9.220376566597205, 'ewma_alpha': 0.396678618590014, 'rolling_window': 5, 'loss_mode': 'rmse'}
  history_params={'ewma_alpha': 0.396678618590014, 'rolling_window': 5}


**`catboost_history`** jointly tunes EWMA alpha, rolling window, CatBoost params, and loss mode (`rmse` vs `multiclass`).
- RMSE: regression with clip for out-of-boundary predictions
- Multiclass: classification

#### `catboost_residual_expanding`


In [47]:
_name = 'catboost_residual_expanding'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    RESIDUAL_ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')
print(f'  params={_params}')
if _result.get('history_params'):
    print(f'  history_params={_result["history_params"]}')


[ok] catboost_residual_expanding  test_mae=0.8423
  params={'iterations': 169, 'depth': 6, 'learning_rate': 0.040528312333610414, 'l2_leaf_reg': 9.98462824624538, 'ewma_alpha': 0.3381906666579405, 'rolling_window': 5}
  history_params={'ewma_alpha': 0.3381906666579405, 'rolling_window': 5}


**`catboost_residual_expanding`**: `pred = clip(expanding_mean + CatBoost_residual)`. Jointly tunes EWMA alpha, rolling window, and CatBoost params (RMSE on residual only).

#### `catboost_history_clf`


In [48]:
_name = 'catboost_history_clf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    HISTORY_CLASSIFICATION_MODELS,
    task='classification',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
classification_results.append(_result)
classification_best_params[_name] = _params
print(f'[ok] {_name}  test_f1={_result["test_metrics"]["f1"]:.4f}')
if _result.get('history_params'):
    print(f'  history_params={_result["history_params"]}')


[ok] catboost_history_clf  test_f1=0.5821
  history_params={'ewma_alpha': 0.4224249300906124, 'rolling_window': 7}


#### `catboost_residual_expanding_clf`

**Formula:** `pred = 1` if `expanding_high_fatigue_rate + CatBoost_residual >= 0.5`, else `0`. Jointly tunes EWMA alpha, rolling window, and CatBoost params.

In [49]:
_name = 'catboost_residual_expanding_clf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    RESIDUAL_CLASSIFICATION_MODELS,
    task='classification',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
classification_results.append(_result)
classification_best_params[_name] = _params
print(f'[ok] {_name}  test_f1={_result["test_metrics"]["f1"]:.4f}')
if _result.get('history_params'):
    print(f'  history_params={_result["history_params"]}')


[ok] catboost_residual_expanding_clf  test_f1=0.5519
  history_params={'ewma_alpha': 0.3740532221732895, 'rolling_window': 2}


## 4. Results summary

Includes baselines plus only models whose §2 cells were executed. Ordinal CV summary shows **`cv_mae_std` only** (fold stability for MAE).

In [50]:
def collect_summaries(results, task='ordinal'):
    cv_rows, test_rows = [], []
    metric_cols = ['mae', 'rmse', 'r2', 'qwk'] if task == 'ordinal' else ['accuracy', 'f1', 'precision', 'recall']
    for result in results:
        cv_mean = result['cv_summary'].loc['mean', metric_cols]
        cv_std = result['cv_summary'].loc['std', metric_cols]
        cv_row = {
            'model': result['name'],
            'best_params': str(result.get('best_params', {})),
        }
        test_row = {
            'model': result['name'],
            'best_params': str(result.get('best_params', {})),
        }
        for col in metric_cols:
            cv_row[f'cv_{col}'] = cv_mean[col]
            test_row[f'test_{col}'] = result['test_metrics'][col]

        if task == 'ordinal':
            cv_row['cv_mae_std'] = cv_std['mae']

        cv_rows.append(cv_row)
        test_rows.append(test_row)
    return pd.DataFrame(cv_rows).set_index('model'), pd.DataFrame(test_rows).set_index('model')

ordinal_results = globals().get('ordinal_results', [])
history_ordinal_results = globals().get('history_ordinal_results', [])
classification_results = globals().get('classification_results', [])
ordinal_best_params = globals().get('ordinal_best_params', {})
history_best_params = globals().get('history_best_params', {})
classification_best_params = globals().get('classification_best_params', {})

ran_ordinal = sorted(set(ordinal_best_params) | set(history_best_params))
ran_classification = sorted(classification_best_params.keys())
print(f'Ran {len(ran_ordinal)} tuned ordinal models: {ran_ordinal}')
print(f'Ran {len(ran_classification)} tuned classification models: {ran_classification}')

all_ordinal_results = ordinal_baseline_results + ordinal_results + history_ordinal_results
all_classification_results = classification_baseline_results + classification_results

ordinal_cv_summary, ordinal_test_summary = collect_summaries(all_ordinal_results, task='ordinal')
clf_cv_summary, clf_test_summary = collect_summaries(all_classification_results, task='classification')

print('Ordinal CV summary (baselines first)')
display(ordinal_cv_summary)
print('Ordinal held-out test summary')
display(ordinal_test_summary)
print('Classification CV summary (baselines first)')
display(clf_cv_summary)
print('Classification held-out test summary')
display(clf_test_summary)


Ran 2 tuned ordinal models: ['catboost_history', 'catboost_residual_expanding']
Ran 2 tuned classification models: ['catboost_history_clf', 'catboost_residual_expanding_clf']
Ordinal CV summary (baselines first)


,best_params,cv_mae,cv_rmse,cv_r2,cv_qwk,cv_mae_std
model,,,,,,
global_mean,{},1.317778,1.552164,-0.088215,0.000000,0.057900
global_mode,{},1.257337,1.601647,-0.157108,0.000000,0.147267
lag1_fatigue,{},0.831127,1.341014,0.177822,0.586093,0.133354
expanding_mean,{},0.917276,1.247144,0.297579,0.519429,0.110738
catboost_history,"{'iterations': 308, 'depth': 4, 'learning_rate...",0.874432,1.174411,0.373974,0.543687,0.121491
catboost_residual_expanding,"{'iterations': 169, 'depth': 6, 'learning_rate...",0.882356,1.223262,0.324819,0.562548,0.089387


Ordinal held-out test summary


,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
global_mean,{},1.321168,1.585979,-0.360095,0.000000
global_mode,{},0.986861,1.372302,-0.018295,0.000000
lag1_fatigue,{},0.934307,1.362694,-0.004086,0.495885
expanding_mean,{},0.839416,1.167066,0.263512,0.458234
catboost_history,"{'iterations': 308, 'depth': 4, 'learning_rate...",0.832117,1.150056,0.284825,0.467497
catboost_residual_expanding,"{'iterations': 169, 'depth': 6, 'learning_rate...",0.842336,1.179509,0.247725,0.469006


Classification CV summary (baselines first)


,best_params,cv_accuracy,cv_f1,cv_precision,cv_recall
model,,,,,
majority_class,{},0.732127,0.000000,0.000000,0.000000
lag1_high_fatigue,{},0.781823,0.579620,0.584633,0.574708
expanding_high_fatigue_rate,{},0.796365,0.526688,0.701580,0.431608
catboost_history_clf,"{'iterations': 218, 'depth': 8, 'learning_rate...",0.762431,0.596639,0.544657,0.668643
catboost_residual_expanding_clf,"{'iterations': 463, 'depth': 4, 'learning_rate...",0.800999,0.558423,0.699351,0.477808


Classification held-out test summary


,best_params,test_accuracy,test_f1,test_precision,test_recall
model,,,,,
majority_class,{},0.680292,0.000000,0.000000,0.000000
lag1_high_fatigue,{},0.705109,0.534562,0.539535,0.529680
expanding_high_fatigue_rate,{},0.747445,0.548303,0.640244,0.479452
catboost_history_clf,"{'iterations': 218, 'depth': 8, 'learning_rate...",0.658394,0.582143,0.478006,0.744292
catboost_residual_expanding_clf,"{'iterations': 463, 'depth': 4, 'learning_rate...",0.741606,0.551899,0.619318,0.497717


## 5. Model Stability

Run each cell separately. Each tuned model is re-fit with full Optuna on every seed. The summary cell adds `expanding_mean` (no tuning) and compares all three. Use `_stability_seeds = [42, 43, 44]` in the first cell for a quick smoke test.

Compare models by `test_mae_mean` directly (lower is better).


In [17]:
# Optional: use fewer seeds for a quick run
# _stability_seeds = [42, 43, 44]
_stability_seeds = STABILITY_SEEDS
stability_parts = globals().get('stability_parts', {})

_stability_history = run_stability_study(
    df,
    seeds=_stability_seeds,
    models=['catboost_history'],
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
stability_parts['catboost_history'] = _stability_history
print(f'[ok] catboost_history stability  rows={len(_stability_history)}')
display(_stability_history)


[ok] catboost_history stability  rows=10


,seed,model,test_mae,test_rmse,test_qwk,cv_mae_mean,cv_mae_std,best_params,n_test_participants
0,42,catboost_history,0.808759,1.133434,0.492327,0.884374,0.123086,"{'iterations': 460, 'depth': 6, 'learning_rate...",8
1,43,catboost_history,0.909091,1.199951,0.493886,0.840080,0.124544,"{'iterations': 228, 'depth': 4, 'learning_rate...",8
2,44,catboost_history,0.758140,1.046959,0.610927,0.889621,0.095686,"{'iterations': 110, 'depth': 4, 'learning_rate...",8
3,45,catboost_history,0.855828,1.151153,0.509164,0.878573,0.187793,"{'iterations': 101, 'depth': 4, 'learning_rate...",8
4,46,catboost_history,0.891374,1.182946,0.418453,0.871336,0.169253,"{'iterations': 212, 'depth': 4, 'learning_rate...",8
5,47,catboost_history,0.925590,1.279462,0.244129,0.851310,0.073251,"{'iterations': 121, 'depth': 4, 'learning_rate...",8
6,48,catboost_history,1.035654,1.296462,0.440160,0.841085,0.101595,"{'iterations': 338, 'depth': 5, 'learning_rate...",8
7,49,catboost_history,0.808896,1.129943,0.677372,0.864997,0.069359,"{'iterations': 273, 'depth': 4, 'learning_rate...",8
8,50,catboost_history,0.910112,1.194557,0.571925,0.845662,0.121724,"{'iterations': 294, 'depth': 4, 'learning_rate...",8
9,51,catboost_history,0.852290,1.117208,0.589757,0.865710,0.056750,"{'iterations': 399, 'depth': 4, 'learning_rate...",8


In [18]:
stability_parts = globals().get('stability_parts', {})

_stability_residual = run_stability_study(
    df,
    seeds=_stability_seeds,
    models=['catboost_residual_expanding'],
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
stability_parts['catboost_residual_expanding'] = _stability_residual
print(f'[ok] catboost_residual_expanding stability  rows={len(_stability_residual)}')
display(_stability_residual)


[ok] catboost_residual_expanding stability  rows=10


,seed,model,test_mae,test_rmse,test_qwk,cv_mae_mean,cv_mae_std,best_params,n_test_participants
0,42,catboost_residual_expanding,0.878832,1.210658,0.437576,0.876267,0.082863,"{'iterations': 426, 'depth': 5, 'learning_rate...",8
1,43,catboost_residual_expanding,0.912023,1.243163,0.505105,0.853264,0.123549,"{'iterations': 173, 'depth': 4, 'learning_rate...",8
2,44,catboost_residual_expanding,0.720930,1.066037,0.669920,0.915832,0.106555,"{'iterations': 348, 'depth': 8, 'learning_rate...",8
3,45,catboost_residual_expanding,0.941718,1.208355,0.482739,0.865735,0.215923,"{'iterations': 365, 'depth': 4, 'learning_rate...",8
4,46,catboost_residual_expanding,1.009585,1.331602,0.283697,0.833948,0.115215,"{'iterations': 109, 'depth': 6, 'learning_rate...",8
5,47,catboost_residual_expanding,0.903811,1.252221,0.250047,0.857204,0.130495,"{'iterations': 147, 'depth': 7, 'learning_rate...",8
6,48,catboost_residual_expanding,1.037351,1.320467,0.446154,0.836696,0.131585,"{'iterations': 104, 'depth': 5, 'learning_rate...",8
7,49,catboost_residual_expanding,0.858320,1.145869,0.731276,0.877619,0.095853,"{'iterations': 101, 'depth': 6, 'learning_rate...",8
8,50,catboost_residual_expanding,0.953451,1.234860,0.589736,0.842714,0.141409,"{'iterations': 405, 'depth': 5, 'learning_rate...",8
9,51,catboost_residual_expanding,0.813885,1.178964,0.594281,0.883858,0.117722,"{'iterations': 334, 'depth': 7, 'learning_rate...",8


In [19]:
stability_parts = globals().get('stability_parts', {})
_seeds = globals().get('_stability_seeds', STABILITY_SEEDS)

_stability_baseline = run_stability_study(
    df,
    seeds=_seeds,
    models=['expanding_mean'],
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)

parts = [_stability_baseline]
for _name in ['catboost_history', 'catboost_residual_expanding']:
    if _name in stability_parts:
        parts.append(stability_parts[_name])

stability_df = pd.concat(parts, ignore_index=True)
stability_summary = summarize_stability(stability_df)

print('Combined per-seed stability results')
display(stability_df)
print('Aggregated stability summary')
display(stability_summary)


Combined per-seed stability results


,seed,model,test_mae,test_rmse,test_qwk,cv_mae_mean,cv_mae_std,best_params,n_test_participants
0,42,expanding_mean,0.836496,1.162052,0.463108,0.916134,0.111049,{},8
1,43,expanding_mean,0.964809,1.261894,0.442902,0.880488,0.162696,{},8
2,44,expanding_mean,0.713178,1.041764,0.653122,0.943297,0.117018,{},8
3,45,expanding_mean,0.914110,1.200716,0.457333,0.894199,0.231135,{},8
4,46,expanding_mean,1.041534,1.354203,0.256255,0.868601,0.116428,{},8
5,47,expanding_mean,0.931034,1.270209,0.210295,0.891549,0.149130,{},8
6,48,expanding_mean,1.096774,1.362859,0.389816,0.855240,0.118089,{},8
7,49,expanding_mean,0.864909,1.153035,0.694839,0.907674,0.104352,{},8
8,50,expanding_mean,0.996790,1.262498,0.518076,0.874565,0.142620,{},8
9,51,expanding_mean,0.859675,1.199458,0.548877,0.910464,0.127777,{},8


Aggregated stability summary


,n_seeds,test_mae_mean,test_mae_std,test_mae_ci95_half,cv_mae_mean,cv_mae_std
model,,,,,,
catboost_history,10,0.875573,0.077693,0.048155,0.863275,0.112304
catboost_residual_expanding,10,0.902991,0.092621,0.057407,0.864314,0.126117
expanding_mean,10,0.921931,0.110788,0.068667,0.894221,0.138029
